> **⚠ This notebook does not work with new (unseen) query batches.**
>
> `SysVI.load_query_data` hard-codes `transfer_batch=False`, raising
> `ValueError` when query contains any batch not in the reference registry.
> sysVI has no scArches surgery path for new batches.
>
> Kept as **scaffolding only**. For real reference mapping use **scPoli**
> (`21_map_scpoli.ipynb`) or **Symphony** (`31_map_symphony.ipynb`).
>
> See `tasks/lessons.md` → "SysVI cannot do reference mapping with new
> (unseen) batch categories (2026-04-19)" for the full diagnosis.

# 11 — Map a query arm onto the sysVI atlas (papermill-parameterized)

Uses scArches surgery: freeze reference encoder, fine-tune query-specific encoder
for a small number of epochs. No X shift needed (sysVI uses Gaussian likelihood).

In [1]:
from pathlib import Path
import sys
import time
import json

sys.path.insert(0, str(Path.cwd().parent))
from src.paths import DATA_OUT, MODEL_OUT
import anndata as ad
from scvi.external import SysVI

/ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_mapping/.pixi/envs/scvi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- PARAMETERS (overridden by papermill) ---
QUERY_SOURCE = "source_8"
T_ARM = "tplus"
MAX_EPOCHS = 20
# --- END PARAMETERS ---

## Load atlas + query

In [3]:
atlas_dir = MODEL_OUT / f"sysvi_atlas_excl_{QUERY_SOURCE}"
query = ad.read_h5ad(DATA_OUT / "query_arms" / f"{QUERY_SOURCE}_{T_ARM}.h5ad")
print(f"atlas: {atlas_dir}, query: {query.shape}")

atlas: /ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_mapping/models/sysvi_atlas_excl_source_8, query: (73797, 1040)


## scArches surgery via SysVI.load_query_data

In [4]:
SysVI.prepare_query_anndata(query, str(atlas_dir))
model_query = SysVI.load_query_data(query, str(atlas_dir))

t0 = time.perf_counter()
model_query.train(
    max_epochs=MAX_EPOCHS,
    plan_kwargs={"weight_decay": 0.0},
)
elapsed = time.perf_counter() - t0
print(f"mapping took {elapsed:.1f}s")

INFO     File                                                                                                      
         /ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_map
         ping/models/sysvi_atlas_excl_source_8/model.pt already downloaded                                         
INFO     Found 100.0% reference vars in query data.                                                                
INFO     File                                                                                                      
         /ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_map
         ping/models/sysvi_atlas_excl_source_8/model.pt already downloaded                                         


/ictstr01/groups/ml01/workspace/ttreis/projects/broad_integrate/2023_Arevalo_BatchCorrection/reference_mapping/.pixi/envs/scvi/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /ictstr01/groups/ml01/workspace/ttreis/projects/broa ...


ValueError: This model does not allow for query having batch categories missing from the reference.

In [ ]:
query.obsm["X_sysvi_mapped"] = model_query.get_latent_representation()

## Persist

In [ ]:
out_dir = DATA_OUT / "mapped"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{QUERY_SOURCE}_{T_ARM}_sysvi.h5ad"
query.write_h5ad(out_path)
(out_dir / f"{QUERY_SOURCE}_{T_ARM}_sysvi_timings.json").write_text(
    json.dumps({"mapping_seconds": elapsed, "max_epochs": MAX_EPOCHS})
)
print(f"wrote {out_path}")